# 05 - Nondivisible satellite signature assembly

This notebook demonstrates the nondivisible branch of the high-level satellite workflow added after the common-`p` calculation in notebook 04. We use the four-fold branched cover of the iterated torus knot `T(2,3;2,5)`. Its outer winding is two, so the cover degree does **not** divide the winding.

The implemented route performs four genuine calculations:

1. it identifies the two companion-cover copies predicted by BCP-II, Theorem 4.19;
2. it transports the satellite character to one character on each copy;
3. it evaluates the matching outer-pattern phase for each transported character; and
4. it computes both lower-cover twisted signature profiles and applies the theorem's phase/power substitutions.

One mathematical ingredient is intentionally **not** claimed: the package does not yet compute the required four-dimensional twisted profile of the outer `T(2,5)` pattern. The assembly example therefore uses a conspicuously labelled synthetic profile. It illustrates the API and propagation of known and unknown contributions, not a completed invariant of this cable.

## 1. Setup

In [ ]:
from pathlib import Path
from pprint import pprint
import sys

repository_root = Path.cwd()
if repository_root.name == "notebooks":
    repository_root = repository_root.parent
source_directory = repository_root / "src"
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

from sage.all import QQ, gcd
from gaknot import (
    BranchedCoverHomology,
    Character,
    GeneralizedAlgebraicKnot,
    SignatureJumpGap,
    TwistedSignatureJumpProfile,
    iterated_torus_nondivisible_signature_jumps,
)

## 2. Identify the branch of Theorem 4.19

Write `n` for the satellite-cover degree and `w` for the winding of the outer pattern. The theorem puts `h=gcd(n,w)`. When `n` does not divide `w`, the companion contribution consists of `h` metabelian forms on the `n/h`-fold cover of the companion, and the variable in each form is replaced by `phase_j * t^(w/h)`.

For this example, `n=4`, `w=2`, `h=2`, the lower cover degree is two, and the substitution exponent is one.

In [ ]:
cover_degree = 4
winding = 2
h = gcd(cover_degree, winding)
lower_cover_degree = cover_degree // h
substitution_exponent = winding // h

print("cover degree n:", cover_degree)
print("outer winding w:", winding)
print("n divides w:", winding % cover_degree == 0)
print("h = gcd(n,w):", h)
print("lower cover degree n/h:", lower_cover_degree)
print("substitution exponent w/h:", substitution_exponent)

## 3. Build the cable and inspect its four-cover homology

The cable sequence is stored from the innermost knot outward: `[(2,3),(2,5)]` means that the `T(2,5)` pattern is applied to the trefoil `T(2,3)`. Branched-cover homology records the resulting layers in the reverse, outside-to-inside order.

The outer pattern occurs once with effective cover degree four. The trefoil layer occurs twice with effective cover degree two. These two copies are the structural source of the two induced characters in Theorem 4.19.

In [ ]:
cable = GeneralizedAlgebraicKnot.iterated_torus_knot(
    [(2, 3), (2, 5)]
)
four_cover = BranchedCoverHomology(cable, cover_degree)

print("knot:", cable)
print("four-cover homology:", four_cover)
print("outer-to-inner decomposition:")
pprint(four_cover.decomposition[0]["layers"])

## 4. Define a character on the satellite cover

Character coordinates follow the same component/layer/generator nesting as the homology decomposition. The outer `T(2,5)` layer has one `Z/5Z` generator, sent below to `1/5`. The inner layer contains two `Z/3Z` copies, sent to `1/3` and `2/3`.

Choosing different values on the two inner copies is useful diagnostically: it makes it impossible to reuse the first induced character for both companion summands without being detected.

In [ ]:
satellite_character = Character(
    four_cover,
    [[[QQ(1) / 5], [QQ(1) / 3, QQ(2) / 3]]],
)

print("flattened character values:", satellite_character.values)
print("outer layer values:",
      satellite_character.restrict_to_layer(0, 0))
print("inner layer copy values:",
      satellite_character.restrict_to_layer(0, 1))

## 5. Transport the character to the companion covers

`induced_companion_characters()` removes the outer cabling stage and restricts the source character to the deck-translated companion copies. Entry `j` is the paper's character `chi_(j+1)` on the `n/h`-fold cover.

The result is an immutable diagnostic object rather than a bare list: it retains the source character, companion knot and homology, theorem indices, deck powers, cover arithmetic, and exact phase orbit.

In [ ]:
transport = satellite_character.induced_companion_characters()

print("companion knot:", transport.companion_knot)
print("companion homology:", transport.companion_homology)
print("theorem indices:", transport.theorem_indices)
print("deck powers:", transport.deck_powers)
print("induced character values:",
      [character.values for character in transport])
print("paired phase arguments:", transport.phase_arguments)

The order of the two tuples is part of the formula: `characters[j]` and `phase_arguments[j]` belong to the same direct-sum term. Here they are `(1/3, 4/5)` and `(2/3, 1/5)`.

## 6. Audit the phase calculation in Smith coordinates

The phase at deck power `j` is the source character evaluated on the distinguished class `t^j q_Q(mu_Q^(-w) eta)`. The package represents each class in exactly the public Smith basis used by `Character`. The orbit record exposes those coordinate vectors, so the convention is independently inspectable rather than hidden inside the final signature calculation.

In [ ]:
phase_orbit = transport.pattern_phase_orbit

print("outer Smith factors:", phase_orbit.smith_factors)
print("outer generator values:", phase_orbit.generator_values)
print("distinguished element coordinates:",
      phase_orbit.distinguished_element_coordinates)

for deck_power, coordinates, phase in zip(
    phase_orbit.deck_powers,
    phase_orbit.smith_coordinates,
    phase_orbit.phase_arguments,
):
    evaluated = sum(
        value * coordinate
        for value, coordinate in zip(
            phase_orbit.generator_values,
            coordinates,
        )
    ) % 1
    print(
        f"deck power {deck_power}: coordinates={coordinates}, "
        f"phase={phase}, direct evaluation={evaluated}"
    )

## 7. Supply the currently external outer-pattern profile

The new high-level function needs the twisted jump profile of the outer pattern in the **original four-dimensional representation**. Yanagida's formulas currently implemented by `gaknot` calculate an `m`-dimensional representation for `T(m,q)`. They therefore provide a two-dimensional profile for `T(2,5)`, not the four-dimensional profile required here. Relabelling the former as cover degree four would be mathematically incorrect.

For the sole purpose of demonstrating assembly, the following object contains invented jump data. Both its variable name and label say `synthetic`. The known jump at `1/7` and the gap at `2/7` are **not claims about `T(2,5)`**. A real application must replace this object with a theorem-backed calculation carrying `cover_degree=4`.

In [ ]:
synthetic_outer_profile = TwistedSignatureJumpProfile(
    known_jumps=((QQ(1) / 7, 2),),
    unresolved=(
        SignatureJumpGap(
            argument=QQ(2) / 7,
            reason=(
                "Synthetic tutorial gap; replace this entire profile with "
                "a genuine four-dimensional outer-pattern calculation."
            ),
            source="synthetic tutorial outer pattern",
        ),
    ),
    cover_degree=4,
    label="SYNTHETIC four-cover T(2,5) tutorial data",
)

print("label:", synthetic_outer_profile.label)
print("known illustrative jumps:", synthetic_outer_profile.known_jumps)
print("illustrative gaps:",
      synthetic_outer_profile.unresolved_arguments)

## 8. Assemble the nondivisible satellite stage

The public function now performs the complete implemented route. Internally it recomputes the transport, invokes the common-`p` Yanagida pipeline once for each induced trefoil character, pairs the resulting profiles with their corresponding phases, and delegates the direct-sum algebra to the implementation of Theorem 4.19.

In [ ]:
assembled = iterated_torus_nondivisible_signature_jumps(
    cable,
    satellite_character,
    synthetic_outer_profile,
)

print("cable sequence:", assembled.cable_sequence)
print("Theorem 4.19 branch:", assembled.satellite_result.case)
print("cover degree, winding, h:", (
    assembled.satellite_result.cover_degree,
    assembled.satellite_result.winding,
    assembled.satellite_result.h,
))
print("induced characters:",
      [character.values for character in assembled.induced_characters])
print("phase arguments:", assembled.phase_arguments)

## 9. Inspect every computed lower-cover result

The two companion calculations are retained separately. Each records the induced character's Smith-to-orbit conversion, Yanagida's local coverage analysis, and the resulting lower-cover profile. The orbit vectors `(2,1)` and `(1,2)` confirm that the two character inputs were not collapsed into one.

For these particular nonzero trefoil characters, every nontrivial-root local contribution is known and zero. Yanagida's universal root-one limitation remains as a gap at argument zero in both raw profiles.

In [ ]:
for index, (character, result, profile) in enumerate(zip(
    assembled.induced_characters,
    assembled.companion_results,
    assembled.companion_profiles,
), start=1):
    print(f"companion term {index}")
    print("  character values:", character.values)
    print("  Yanagida orbit:", result.orbit.a_values)
    print("  profile cover degree:", profile.cover_degree)
    print("  known jumps:", profile.known_jumps)
    print("  unresolved arguments:", profile.unresolved_arguments)

## 10. Follow the phase/power substitutions

Here `w/h=1`. A raw companion gap at argument zero pulls back through `phase_j * t` to argument `-phase_j mod 1`. The phases `4/5` and `1/5` therefore move the two root-one gaps to `1/5` and `4/5`, in that order.

This is why a root that was `t=1` in a lower-cover calculation can become a nontrivial unresolved root after satellite assembly. The software propagates that uncertainty instead of interpreting it as a zero contribution.

In [ ]:
for index, (phase, raw, transformed) in enumerate(zip(
    assembled.phase_arguments,
    assembled.companion_profiles,
    assembled.companion_summands,
), start=1):
    print(f"companion term {index}")
    print("  phase:", phase)
    print("  raw gaps:", raw.unresolved_arguments)
    print("  transformed gaps:", transformed.unresolved_arguments)

print("assembled known jumps:", assembled.total_profile.known_jumps)
print("all assembled gaps:", assembled.unresolved_arguments)
print("complete profile:", assembled.is_complete)

The total has three explicit gaps: the synthetic outer gap at `2/7`, plus the genuine propagated companion gaps at `1/5` and `4/5`. The known illustrative outer jump at `1/7` remains unchanged. Contributions from different summands are never silently cancelled merely because some of them are unknown.

## 11. Coverage-aware failure and dimensional validation

`known_jump_at(x)` returns the proved portion even if another direct-sum term is unresolved there. `jump_at(x)` requires the complete local answer and raises `NotImplementedError` at a gap. The outer profile's representation degree is also checked before any assembly: a two-cover pattern profile cannot be used in this four-cover calculation.

In [ ]:
print("known part at 1/5:",
      assembled.total_profile.known_jump_at(QQ(1) / 5))

try:
    assembled.total_profile.jump_at(QQ(1) / 5)
except NotImplementedError as error:
    print("complete local jump unavailable:", error)

wrong_dimension = TwistedSignatureJumpProfile(
    cover_degree=2,
    label="wrong-dimensional example",
)
try:
    iterated_torus_nondivisible_signature_jumps(
        cable,
        satellite_character,
        wrong_dimension,
    )
except ValueError as error:
    print("dimension check:", error)

## 12. Current scope and next mathematical step

The completed software now knows how to transport characters, compute exact phase arguments, evaluate supported lower-cover profiles, perform the nondivisible substitutions, and preserve all coverage gaps. Its bounded high-level interface requires the remaining companion to be a positive common-`p` iterated torus knot whose induced cover degree equals `p`.

The principal missing input is a genuine outer-pattern profile when the representation dimension differs from the torus pattern's first parameter. The next research step is to determine whether Yanagida's matrix construction can be generalized from the implemented `T(m,q)`/dimension-`m` setting to `T(w,q)` in dimension `n` with `n != w`. Until that is justified, the API keeps the outer profile explicit.

Main references:

- Borodzik--Conway--Politarczyk, *Twisted Blanchfield pairings and twisted signatures II: Relation to Casson--Gordon invariants*, [arXiv:1809.08791](https://arxiv.org/abs/1809.08791), especially Theorem 4.19;
- Koki Yanagida, *Blanchfield pairings and twisted Blanchfield pairings of torus knots*, [arXiv:2602.07575v2](https://arxiv.org/abs/2602.07575v2); and
- Conway--Kim--Politarczyk, *Non-slice linear combinations of iterated torus knots*, [arXiv:1910.01368](https://arxiv.org/abs/1910.01368), for the character-orbit conventions used in these examples.

## Exercises

1. Replace the two inner values by `1/3, 1/3`. Which diagnostic fields become equal, and which theorem terms must still remain separately present?
2. Change only the outer `Z/5Z` character value from `1/5` to `2/5` and inspect the Smith-coordinate phase evaluations.
3. Add another synthetic gap at `1/5`, assemble again, and inspect the two distinct unresolved sources at the same argument. Explain why they cannot be cancelled without computing both local pairings.
4. Construct `T(3,4;2,5)` in the four-cover. Verify that character transport succeeds but high-level companion-profile computation stops because the induced double cover does not match the inner first parameter `p=3`.